In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models


# ==========================================
# 1. DYNAMIC MULTI-SCALE RESNET BACKBONE
# ==========================================
class ResNetBackbone(nn.Module):
    """
    Extracts multi-scale 4D features dynamically from any standard ResNet.
    Extracts Layer 2 (P3), Layer 3 (P4), and Layer 4 (P5).
    """

    def __init__(self, resnet_name="resnet34", pretrained=True):
        super().__init__()
        # Load the base model dynamically
        weights = "DEFAULT" if pretrained else None
        base_model = getattr(models, resnet_name)(weights=weights)

        # Base stem
        self.stem = nn.Sequential(
            base_model.conv1, base_model.bn1, base_model.relu, base_model.maxpool
        )
        # Feature layers
        self.layer1 = base_model.layer1  # P2 (not used)
        self.layer2 = base_model.layer2  # P3 (Stride 8)
        self.layer3 = base_model.layer3  # P4 (Stride 16)
        self.layer4 = base_model.layer4  # P5 (Stride 32)

        # DYNAMIC CHANNEL DISCOVERY:
        # Dummy pass to calculate out_channels automatically for any ResNet flavor
        self.out_channels = self._get_out_channels()

    def _get_out_channels(self):
        self.eval()
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 224, 224)
            x = self.stem(dummy)
            p1 = self.layer1(x)
            p3 = self.layer2(p1)
            p4 = self.layer3(p3)
            p5 = self.layer4(p4)
        return [p3.shape[1], p4.shape[1], p5.shape[1]]

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        p3 = self.layer2(x)  # e.g., ResNet34: [B, 128, H/8, W/8]
        p4 = self.layer3(p3)  # e.g., ResNet34: [B, 256, H/16, W/16]
        p5 = self.layer4(p4)  # e.g., ResNet34: [B, 512, H/32, W/32]
        return [p3, p4, p5]


# ==========================================
# 2. NECK (FPN for Channel Standardization)
# ==========================================
class SimpleNeckFPN(nn.Module):
    """
    Takes arbitrary multi-scale channels from ResNet and standardizes
    them to a fixed projection dimension (e.g., 256 channels) while fusing features.
    """

    def __init__(self, in_channels, out_chan=256):
        super().__init__()
        # Projections to unify channels
        self.p5_proj = nn.Conv2d(in_channels[2], out_chan, kernel_size=1)
        self.p4_proj = nn.Conv2d(in_channels[1], out_chan, kernel_size=1)
        self.p3_proj = nn.Conv2d(in_channels[0], out_chan, kernel_size=1)

        # Upsampling block
        self.up = nn.Upsample(scale_factor=2, mode="nearest")

    def forward(self, features):
        p3, p4, p5 = features

        # Top-down pathway
        f5 = self.p5_proj(p5)
        f4 = self.p4_proj(p4) + self.up(f5)
        f3 = self.p3_proj(p3) + self.up(f4)

        return [f3, f4, f5]


# ==========================================
# 3. YOLOV8-LIKE DECOUPLED ANCHOR-FREE HEAD
# ==========================================
class YOLOv8DecoupledHead(nn.Module):
    """
    An anchor-free, decoupled head that splits classification (Cls)
    and regression (Box) predictions per scale grid cell.
    """

    def __init__(self, num_classes, in_channels=256, reg_max=16):
        super().__init__()
        self.num_classes = num_classes
        self.reg_max = (
            reg_max  # YOLOv8 uses DFL (Distribution Focal Loss) bounded by reg_max
        )

        # Decoupled Sub-branches for Box Reg & Classification
        self.cls_convs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(in_channels, in_channels, 3, padding=1),
                    nn.BatchNorm2d(in_channels),
                    nn.SiLU(),
                    nn.Conv2d(in_channels, in_channels, 3, padding=1),
                    nn.BatchNorm2d(in_channels),
                    nn.SiLU(),
                )
                for _ in range(3)
            ]
        )

        self.reg_convs = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(in_channels, in_channels, 3, padding=1),
                    nn.BatchNorm2d(in_channels),
                    nn.SiLU(),
                    nn.Conv2d(in_channels, in_channels, 3, padding=1),
                    nn.BatchNorm2d(in_channels),
                    nn.SiLU(),
                )
                for _ in range(3)
            ]
        )

        # Final predictions per grid cell
        # Class probabilities branch
        self.cls_preds = nn.ModuleList(
            [nn.Conv2d(in_channels, num_classes, 1) for _ in range(3)]
        )
        # Bounding box branch: predicts top, left, bottom, right distances scaled by reg_max
        self.reg_preds = nn.ModuleList(
            [nn.Conv2d(in_channels, 4 * reg_max, 1) for _ in range(3)]
        )

    def forward(self, x):
        outputs = []
        for i in range(3):  # For P3, P4, P5 scales
            features = x[i]

            cls_feat = self.cls_convs[i](features)
            reg_feat = self.reg_convs[i](features)

            cls_out = self.cls_preds[i](cls_feat)  # [B, num_classes, H_i, W_i]
            reg_out = self.reg_preds[i](reg_feat)  # [B, 4 * reg_max, H_i, W_i]

            outputs.append((cls_out, reg_out))
        return outputs


# ==========================================
# 4. UNIFIED UPSTREAM DETECTOR
# ==========================================
class UniversalResNetYOLO(nn.Module):
    def __init__(self, resnet_name="resnet34", num_classes=20, pretrained=True):
        super().__init__()
        # 1. Swap backbones dynamically
        self.backbone = ResNetBackbone(resnet_name=resnet_name, pretrained=pretrained)

        # 2. Automatically pass backbone's discovered channel sizes to Neck
        self.neck = SimpleNeckFPN(in_channels=self.backbone.out_channels, out_chan=256)

        # 3. Static Decoupled Head
        self.head = YOLOv8DecoupledHead(num_classes=num_classes, in_channels=256)

    def forward(self, x):
        feats = self.backbone(x)
        fused_feats = self.neck(feats)
        predictions = self.head(fused_feats)
        return predictions


In [4]:
# --- Test Scenario 1: Using ResNet-34 ---
model_r34 = UniversalResNetYOLO(resnet_name="resnet34", num_classes=80)
print("ResNet34 Backbone Outputs Channels:", model_r34.backbone.out_channels)
# Expected Output: [128, 256, 512]

# # --- Test Scenario 2: Drop-in Replace with ResNet-50 ---
# model_r50 = UniversalResNetYOLO(resnet_name="resnet50", num_classes=80)
# print("ResNet50 Backbone Outputs Channels:", model_r50.backbone.out_channels)
# # Expected Output: [512, 1024, 2048]

# # --- Verification of Output Tensors ---
# dummy_image = torch.randn(1, 3, 640, 640)  # Standard YOLO resolution
# outputs = model_r50(dummy_image)

# for idx, (cls_map, reg_map) in enumerate(outputs):
#     print(f"Scale P{idx + 3} -> Cls Map: {cls_map.shape}, Reg Map: {reg_map.shape}")


ResNet34 Backbone Outputs Channels: [128, 256, 512]


In [6]:
model_r34(torch.randn(1, 3, 640, 640))

TypeError: expected Tensor as element 0 in argument 0, but got tuple

Running validation setup on environment device target: cpu

Executing forward tracking pass and NMS post-processing pipeline...
Feature maps shape: torch.Size([2, 512, 7, 7])
Pooled features shape (after RoI Align): torch.Size([4, 512, 7, 7])
Flattened pooled features shape: torch.Size([4, 25088])
Head outputs shape: torch.Size([4, 1024])
Class logits shape: torch.Size([4, 4])
BBox deltas shape: torch.Size([4, 16])

--- Results for Image 0 ---
Detected Bounding Boxes shape:  torch.Size([3, 4])
Confidence Scores output:       [0.9999645 0.9999596 0.6008893]
Predicted Class Labels output:  [0 0 2]

--- Results for Image 1 ---
Detected Bounding Boxes shape:  torch.Size([1, 4])
Confidence Scores output:       [0.99998224]
Predicted Class Labels output:  [0]
